# Indian Legal RAG - Fine-Tuning with LoRA

This notebook fine-tunes a **Mistral 7B** (or LLaMA-2) model to answer Indian legal queries with strict citation format.

**Goal**:
- Improve structural quality of answers.
- Enforce citation: "According to Section X of Act Y..."
- Reduce hallucinations.

**Note**: This notebook is designed to run on a T4 GPU (free tier) or A100.

## 1. Environment Setup

In [ ]:
!pip install -q -U torch transformers datasets peft bitsandbytes trl accelerate

In [ ]:
import torch
from datasets import load_dataset, Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from trl import SFTTrainer
import json
import os

# Check GPU
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Execution will be slow.")

## 2. Dataset Loading & Validation
Upload your `*_qa.json` files (e.g., `ipc_qa.json`) to the Colab runtime.

In [ ]:
# Load dataset - Change filename as needed
files = ['ipc_qa.json', 'crpc_qa.json', 'constitution_qa.json']
data = []

for f_name in files:
    if os.path.exists(f_name):
        with open(f_name, 'r', encoding='utf-8') as f:
            try:
                file_data = json.load(f)
                # Ensure list of dicts
                if isinstance(file_data, list):
                    data.extend(file_data)
                else:
                    print(f"Skipping {f_name}: Not a JSON list")
            except Exception as e:
                print(f"Error reading {f_name}: {e}")
    else:
        print(f"File not found: {f_name} (Upload it to Colab)")

print(f"Total raw samples: {len(data)}")

# Simple Validation Schema
clean_data = []
for item in data:
    if 'instruction' in item and 'response' in item:
        # Basic filter for short/empty answers
        if len(item['response'].strip()) > 10:
            clean_data.append(item)

print(f"Valid samples after filtering: {len(clean_data)}")
print("Sample:", clean_data[0] if clean_data else "None")

## 3. Prompt Formatting
We convert the data into an instruction format suitable for Mistral/LLaMA.
We enforced the structure: `According to Section [X] of [Act], ...`

In [ ]:
def format_instruction(sample):
    # Standard Alpaca/Instruction format
    return f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['response']}"

dataset = Dataset.from_list(clean_data)
print("Formatted Sample:\n", format_instruction(dataset[0]))

## 4. Model Loading (Quantized)
Loading **Mistral 7B v0.1** in 4-bit precision to fit in T4 memory.

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## 5. LoRA Configuration
Targeting `q_proj` and `v_proj` as requested.

In [ ]:
peft_config = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=16,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"] # Can add k_proj, o_proj for better performance
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

## 6. Training
Training for limited epochs to avoid overfitting.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text", # SFTTrainer expects a text field if formatting function not passed directly
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=format_instruction
)

trainer.train()

## 7. Evaluation & Inference
Testing the model on a few unseen queries.

In [ ]:
def ask_legal_bot(query, model, tokenizer):
    prompt = f"### Instruction:\n{query}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
        top_k=50,
        top_p=0.95
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test
test_q = "What is the punishment for robbery?"
print(ask_legal_bot(test_q, model, tokenizer))

## 8. Saving Adapters
Save the fine-tuned LoRA adapters for use in your RAG pipeline.

In [ ]:
new_model = "mistral-7b-indian-law-lora"
trainer.model.save_pretrained(new_model)
trainer.tokenizer.save_pretrained(new_model)

print(f"Model adapters saved to {new_model}")
# ZIP and download if needed
!zip -r mistral_law_lora.zip mistral-7b-indian-law-lora